In [ ]:
import os
from pathlib import Path
import requests
from PIL import Image
import random
import csv
import re

### Functions: 

In [ ]:

def fetch_page(url):
    """Fetch a single page from the STAC API. Returns parsed JSON as a dict."""
    # Send a GET request to the given URL
    response = requests.get(url)
    # Raise an error if the request failed (e.g. 404, 500)
    response.raise_for_status()
    # Parse the response body as JSON and return it
    return response.json()

In [ ]:
def fetch_all_items(collection_url):
    """Fetch all items from a STAC collection by paginating through all pages.
    Returns a list of all items (features) across all pages."""
    
    all_items = []
    # Start with the first page
    url = collection_url
    
    while url:
        # Fetch the current page
        page = fetch_page(url)
        # Add the features from this page to our list
        all_items.extend(page['features'])
        print(f"Fetched {len(all_items)} items so far...")
        
        # Look for a 'next' link to get the URL of the next page
        # If there is no 'next' link, the while loop will end
        url = None
        for link in page['links']:
            if link['rel'] == 'next':
                url = link['href']
                break
    
    print(f"Done. Total items fetched: {len(all_items)}")
    return all_items
    

In [ ]:
def extract_hrefs(items, variant):
    """Extract (item_id, href, variant) tuples from a list of items.
    
    variant can be 'kgrs', 'komb', 'krel', or 'all'.
    'all' returns one entry per variant per item (three entries per item).
    Returns a list of tuples: (item_id, href, variant)."""
    
    valid_variants = ['kgrs', 'komb', 'krel']
    
    # Determine which variants to extract
    if variant == 'all':
        variants_to_extract = valid_variants
    elif variant in valid_variants:
        variants_to_extract = [variant]
    else:
        raise ValueError(f"Invalid variant '{variant}'. Choose from: kgrs, komb, krel, all")
    
    result = []
    for item in items:
        item_id = item['id']
        for asset_key, asset_value in item['assets'].items():
            # Check the geoadmin:variant field to match the desired variant(s)
            if asset_value.get('geoadmin:variant') in variants_to_extract:
                href = asset_value['href']
                asset_variant = asset_value['geoadmin:variant']
                result.append((item_id, href, asset_variant))
    
    return result

In [ ]:

def download_map(item_id, href, variant, output_dir):
    """Download a single GeoTIFF from the given URL and save it to output_dir.
    The filename is derived from the item_id and variant.
    Skips the download if the file already exists."""
    
    # Build the output filename and full path
    filename = f"{item_id}_{variant}.tif"
    output_path = os.path.join(output_dir, filename)
    
    # Skip if already downloaded (useful if the process is interrupted)
    if os.path.exists(output_path):
        print(f"Already exists, skipping: {filename}")
        return output_path
    
    # Stream the download to avoid loading the entire file into memory at once
    response = requests.get(href, stream=True)
    response.raise_for_status()
    
    with open(output_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    
    print(f"Downloaded: {filename}")
    return output_path

In [ ]:
def download_all_maps(hrefs, output_dir):
    """Download all maps from a list of (item_id, href, variant) tuples.
    Saves each file to output_dir. Skips files that already exist."""
    
    total = len(hrefs)
    
    for i, (item_id, href, variant) in enumerate(hrefs, start=1):
        print(f"[{i}/{total}] Downloading {item_id}_{variant}...")
        download_map(item_id, href, variant, output_dir)
    
    print(f"All done. {total} files processed.")

In [ ]:
def is_white_patch(image, white_threshold=0.5, brightness_cutoff=240):
    """Check if a patch is predominantly white and should be discarded.
    
    Args:
        image (PIL.Image): The patch to check
        white_threshold (float): Fraction of pixels that must be white to discard (default 0.5)
        brightness_cutoff (int): Pixel value above which a pixel is considered white (default 240)
    
    Returns:
        bool: True if the patch is predominantly white, False otherwise
    """
    # Convert to greyscale to simplify pixel comparison
    grey = image.convert('L')
    
    # Count pixels above the brightness cutoff
    pixels = grey.getdata()
    white_pixels = sum(1 for p in pixels if p >= brightness_cutoff)
    
    # Calculate the fraction of white pixels
    white_fraction = white_pixels / len(pixels)
    
    return white_fraction >= white_threshold

In [ ]:

def extract_patches(image, patch_size, num_patches):
    """Extract patches from a large image using a grid, then subsample if needed.
    White patches are discarded.
    
    Args:
        image (PIL.Image): The source image to extract patches from
        patch_size (int): The side length of each square patch in pixels
        num_patches (int): Maximum number of patches to return per image
    
    Returns:
        list: List of PIL.Image patches
    """
    width, height = image.size
    patches = []
    
    # Generate all non-overlapping grid patches
    for top in range(0, height - patch_size + 1, patch_size):
        for left in range(0, width - patch_size + 1, patch_size):
            patch = image.crop((left, top, left + patch_size, top + patch_size))
            
            # Discard predominantly white patches
            if not is_white_patch(patch):
                patches.append(patch)
    
    # If we have more patches than needed, randomly subsample
    if len(patches) > num_patches:
        patches = random.sample(patches, num_patches)
    
    return patches

In [ ]:
def save_patches(patches, item_id, variant, output_dir, start_index):
    """Save a list of patches as JPG files to output_dir.
    
    Args:
        patches (list): List of PIL.Image patches
        item_id (str): The STAC item id
        variant (str): The map variant (kgrs, komb, krel)
        output_dir (str): Directory to save the patches
        start_index (int): Global patch counter to ensure unique filenames
    
    Returns:
        list: List of (patch_id, file_path) tuples
    """
    result = []
    for i, patch in enumerate(patches):
        global_index = start_index + i
        patch_id = f"{item_id}_{variant}_patch{global_index}"
        filename = f"{patch_id}.jpg"
        output_path = os.path.join(output_dir, filename)
        patch.save(output_path, format='JPEG')
        result.append((patch_id, output_path))
    return result



In [ ]:
def process_tile(tif_path, item_id, variant, output_dir, patch_size, num_patches, start_index):
    img = Image.open(tif_path)
    img = img.convert('RGB')
    patches = extract_patches(img, patch_size, num_patches)
    print(f"Extracted {len(patches)} patches from {item_id}_{variant}")
    saved = save_patches(patches, item_id, variant, output_dir, start_index)
    return saved

In [ ]:
def process_all_tiles(tif_dir, output_dir, patch_size, num_patches, label=1):
    """Process all GeoTIFF tiles, extract patches, save as JPG, write labels.csv."""
    os.makedirs(output_dir, exist_ok=True)
    tif_files = [f for f in os.listdir(tif_dir) if f.endswith('.tif')]
    print(f"Found {len(tif_files)} tiles to process")
    all_patches = []
    global_counter = 1
    for i, tif_file in enumerate(tif_files, start=1):
        tif_path = os.path.join(tif_dir, tif_file)
        stem = tif_file.replace('.tif', '')
        parts = stem.rsplit('_', 1)
        item_id = parts[0]
        variant = parts[1]
        print(f"[{i}/{len(tif_files)}] Processing {stem}...")
        try:
            saved = process_tile(tif_path, item_id, variant, output_dir, patch_size, num_patches, global_counter)
            all_patches.extend(saved)
            global_counter += len(saved)
        except Exception as e:
            print(f"  Skipping {stem}: {e}")
            continue
    csv_path = os.path.join(output_dir, 'labels.csv')
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['image_id', 'file_paths', 'label'])
        for patch_id, file_path in all_patches:
            match = re.search(r'(\d+)$', patch_id)
    
            if match:
                id_int = int(match.group(1))
            else:
                print('Error: No integer at the end of patch_id')
                break
            writer.writerow([id_int, file_path, label])
    print(f"\nDone. {len(all_patches)} patches saved.")
    print(f"labels.csv written to {csv_path}")
    return csv_path

In [ ]:
import sys
print(sys.executable)

### Define file paths:

In [ ]:

project_dir = Path(os.getcwd())

root_path = (project_dir / '..' / 'data_folders').resolve()

print(os.listdir(root_path))

data_path_orig = root_path / 'swiss_topo_data_orig'
data_path_orig

### Get all items to download:

In [ ]:
collection_url = "https://data.geo.admin.ch/api/stac/v1/collections/ch.swisstopo.pixelkarte-farbe-pk25.noscale/items"
all_items = fetch_all_items(collection_url)

In [ ]:
print(type(all_items))
print(len(all_items))
print(type(all_items[600]))
print(all_items[600].keys())
print(type(all_items[600]['assets']))
print(all_items[600]['assets'].keys())

### Extract hypertext-references (hrefs) from items:

In [ ]:
hrefs = extract_hrefs(all_items, 'all')
print(f"Total hrefs extracted: {len(hrefs)}")
print(hrefs[0])

In [ ]:
print(hrefs[1])
print(hrefs[2])
print(hrefs[3])

### Download and save image files:

In [ ]:
# Test with just the first 7 hrefs
output_dir = data_path_orig
download_all_maps(hrefs, output_dir)

In [ ]:
img_path = data_path_orig / 'swiss-map-raster25_1984_1056_kgrs.tif'

#img = Image.open(os.path.join(output_dir, "swiss-map-raster25_1984_1056_kgrs.tif"))
img = Image.open(img_path)
display(img)
#img = img.convert('RGB')
#img.show()
#print(is_white_patch(img))


In [ ]:
img = Image.open(data_path_orig / 'swiss-map-raster25_1984_1056_kgrs.tif')
img = img.convert('RGB')
patches = extract_patches(img, patch_size=320, num_patches=10)
print(f"Number of patches extracted: {len(patches)}")
display(patches[0])

In [ ]:
csv_path = process_all_tiles(
    tif_dir=str(data_path_orig),
    output_dir=str(data_path_orig / 'patches'),
    patch_size=320,
    num_patches=10
)

In [ ]:
data_path_orig_patches = data_path_orig / 'patches'

# file_list = os.listdir(data_path_orig_patches)

file_list = sorted(os.listdir(data_path_orig_patches), key=lambda f: os.path.getmtime(os.path.join(data_path_orig_patches, f)))
print(file_list[0])
print(file_list[2])
print(file_list[-2])
print(file_list[-1])
print(len(file_list))

In [ ]:
labels_path = data_path_orig_patches / 'labels.csv'
labels_path 

In [ ]:
import pandas as pd
labels_path
df = pd.read_csv(labels_path)
df.head()


In [ ]:
#df = pd.read_csv(labels_path)

# Extract trailing numbers from image_id and filename
#id_numbers = df['image_id'].str.extract(r'(\d+)$')[0].astype(int)
id_numbers = df['image_id']
filename_numbers = df['file_paths'].apply(lambda x: re.search(r'(\d+)\.jpg$', x).group(1)).astype(int)

# Check uniqueness
print("image_id numbers unique:", id_numbers.nunique() == len(df))
print("filename numbers unique:", filename_numbers.nunique() == len(df))

# Check they match
print("image_id and filename numbers match:", (id_numbers == filename_numbers).all())



In [ ]:

missing = df[~df['file_paths'].apply(os.path.exists)]
print(f"{len(missing)} missing files out of {len(df)}")
print(missing['file_paths'].head(10))
